# 1. Loading the models and unseen test datasets

In [1]:
%%time
'''---------Import necessary libraries-----------'''
import pandas as pd
import numpy as np
import scipy.sparse as sp
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, precision_recall_curve, roc_curve, auc,
    jaccard_score
)
from skopt.space import Integer, Real
import matplotlib.pyplot as plt
import joblib
import os
import dash
from dash import dcc, html, Input, Output, dash_table
import plotly.express as px
import plotly.graph_objects as go

# Uncomment if you don't have these installed already!
'''
!pip install dash jupyter-dash plotly pandas
!pip install jupyter-dash
'''
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
from dash.exceptions import PreventUpdate
import warnings
warnings.filterwarnings("ignore")
'''---------Importing the pretrained models and unseen test datasets-----------'''
'''------------------------10 features-----------------------------------------'''
# Define the base path for 10 features
base_path_10 = os.path.join('..', '..', 'data', 'processed', '10_rfe')

# Load the models and data with the specified path for 50 features
base_xgb_model_10 = joblib.load(os.path.join(base_path_10, "final_model_10_features.joblib"))
best_xgb_model_ADASYN_10 = joblib.load(os.path.join(base_path_10, "best_xgb_model_ADASYN_10.joblib"))
stack_model_10 = joblib.load(os.path.join(base_path_10, "stack_model_10.joblib"))
stack_model_adasyn_10 = joblib.load(os.path.join(base_path_10, "stack_model_ADASYN_10.joblib"))

'''------------------------20 features-----------------------------------------'''
# Define the base path for 20 features
base_path_20 = os.path.join('..', '..', 'data', 'processed', '20_rfe')

# Load the models and data with the specified path for 20 features
base_xgb_model_20 = joblib.load(os.path.join(base_path_20, "final_model_20_features.joblib"))
best_xgb_model_ADASYN_20 = joblib.load(os.path.join(base_path_20, "best_xgb_model_ADASYN.joblib"))
stack_model_20 = joblib.load(os.path.join(base_path_20, "stack_model.joblib"))
stack_model_adasyn_20 = joblib.load(os.path.join(base_path_20, "stack_model_ADASYN.joblib"))
'''------------------------50 features-----------------------------------------'''
# Define the base path for 50 features
base_path_50 = os.path.join('..', '..', 'data', 'processed', '50_rfe')

# Load the models and data with the specified path for 50 features
base_xgb_model_50 = joblib.load(os.path.join(base_path_50, "xgb_model_50.joblib"))
best_xgb_model_ADASYN_50 = joblib.load(os.path.join(base_path_50, "best_xgb_model_ADASYN_50.joblib"))
stack_model_50 = joblib.load(os.path.join(base_path_50, "stack_model_50.joblib"))
stack_model_adasyn_50 = joblib.load(os.path.join(base_path_50, "stack_model_ADASYN_50.joblib"))

'''--------------------Unseen Test Datasets------------------------------------'''
# Load Test Unseen datasets
X_unseen_rfe_10, y_unseen_10 = joblib.load(os.path.join(base_path_10, 'X_unseen_rfe_y_unseen_10.joblib'))
X_unseen_rfe_20, y_unseen_20 = joblib.load(os.path.join(base_path_20, 'X_unseen_rfe_y_unseen.joblib'))
X_unseen_rfe_50, y_unseen_50 = joblib.load(os.path.join(base_path_50, 'X_unseen_rfe_y_unseen.joblib'))

CPU times: total: 6.55 s
Wall time: 2.25 s


# 2. Predicting on Unseen test datasets using different models at different thresholds to optimize Precision, Recall, F1 score, Jaccard score, etc..

In [2]:
%%time
# Define a function to evaluate on each model
def evaluate_model_on_unseen_data(model, X_unseen, y_unseen, model_name):
    y_unseen_pred_proba = model.predict_proba(X_unseen)[:, 1]
    
    thresholds = np.append(np.arange(0.1, 0.9, 0.1), [0.02, 0.9, 0.95, 0.98, 0.99])
    thresholds = np.sort(thresholds)

    metrics_list = []
    fp_cost = 10
    fn_cost = 100
    tp_savings = 100

    for threshold in thresholds:
        y_pred = (y_unseen_pred_proba >= threshold).astype(int)
        
        if len(np.unique(y_pred)) == 1:
            continue
        
        precision = precision_score(y_unseen, y_pred, zero_division=0)
        recall = recall_score(y_unseen, y_pred, zero_division=0)
        f1 = f1_score(y_unseen, y_pred, zero_division=0)
        roc_auc = roc_auc_score(y_unseen, y_unseen_pred_proba)
        jaccard = jaccard_score(y_unseen, y_pred, zero_division=0)
        
        conf_matrix = confusion_matrix(y_unseen, y_pred)
        if conf_matrix.shape != (2, 2):
            continue
        
        tn, fp, fn, tp = conf_matrix.ravel()
        
        total_savings = tp * tp_savings
        cost_of_operation = (fp * fp_cost) + (fn * fn_cost)
        net_savings = total_savings - cost_of_operation
        
        metrics_list.append({
            'Model Name': model_name,
            'Threshold': threshold,
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1,
            'ROC AUC': roc_auc,
            'Jaccard Score': jaccard,
            'True Positives (TP)': tp,
            'False Positives (FP)': fp,
            'True Negatives (TN)': tn,
            'False Negatives (FN)': fn,
            'Total Savings ($)': total_savings,
            'Cost of Operation ($)': cost_of_operation,
            'Net Savings ($)': net_savings
        })

    return pd.DataFrame(metrics_list)

# Define the list of models and their corresponding unseen datasets
models_and_data = [
    (base_xgb_model_10, X_unseen_rfe_10, y_unseen_10, 'Base XGBoost (10 Features)'),
    (best_xgb_model_ADASYN_10, X_unseen_rfe_10, y_unseen_10, 'Tuned XGBoost ADASYN (10 Features)'),
    (stack_model_10, X_unseen_rfe_10, y_unseen_10, 'Base Stacking Classifiers (10 Features)'),
    (stack_model_adasyn_10, X_unseen_rfe_10, y_unseen_10, 'Tuned Stacking Classifiers ADASYN (10 Features)'),
    (base_xgb_model_20, X_unseen_rfe_20, y_unseen_20, 'Base XGBoost Model (20 Features)'),
    (best_xgb_model_ADASYN_20, X_unseen_rfe_20, y_unseen_20, 'Tuned XGBoost ADASYN (20 Features)'),
    (stack_model_20, X_unseen_rfe_20, y_unseen_20, 'Base Stacking Classifiers (20 Features)'),
    (stack_model_adasyn_20, X_unseen_rfe_20, y_unseen_20, 'Tuned Stacking Classifiers ADASYN (20 Features)'),
    (base_xgb_model_50, X_unseen_rfe_50, y_unseen_50, 'Base XGBoost (50 Features)'),
    (best_xgb_model_ADASYN_50, X_unseen_rfe_50, y_unseen_50, 'Tuned XGBoost ADASYN (50 Features)'),
    (stack_model_50, X_unseen_rfe_50, y_unseen_50, 'Base Stacking Classifiers (50 Features)'),
    (stack_model_adasyn_50, X_unseen_rfe_50, y_unseen_50, 'Tuned Stacking Classifiers ADASYN (50 Features)')
]

all_metrics_df_list = []

# Evaluate each model and store the results
for model, X_unseen, y_unseen, model_name in models_and_data:
    metrics_df = evaluate_model_on_unseen_data(model, X_unseen, y_unseen, model_name)
    all_metrics_df_list.append(metrics_df)

# Combine all results into a single DataFrame
all_metrics_df = pd.concat(all_metrics_df_list, ignore_index=True)

# Filter the best rows based on different criteria
best_net_savings = all_metrics_df.loc[all_metrics_df.groupby('Model Name')['Net Savings ($)'].idxmax()]
best_f1 = all_metrics_df.loc[all_metrics_df.groupby('Model Name')['F1 Score'].idxmax()]
balanced_prec_recall = all_metrics_df[(all_metrics_df['Precision'] >= 0.5)].copy()

# Find rows where the difference between Precision and Recall is minimal
balanced_prec_recall = balanced_prec_recall.loc[balanced_prec_recall.groupby('Model Name')['Recall'].idxmax()]

# Combine the best results into a single table
best_results = pd.concat([best_net_savings, best_f1, balanced_prec_recall]).drop_duplicates().reset_index(drop=True)
# Save the DataFrame to a CSV file
all_metrics_df.to_csv('all_results.csv', index=False)
best_results.to_csv('best_results.csv', index=False)

CPU times: total: 2min 23s
Wall time: 1min 42s


# 3. Visualising top 3 models on a custom metrics on an enteractive dashboard

In [3]:
# load the saved all_metrics Dataframe
all_results = pd.read_csv('all_results.csv')
# Load the saved best_results DataFrame
best_results = pd.read_csv('best_results.csv')

# Round the specific columns to 2 decimal places
best_results['Precision'] = best_results['Precision'].round(2)
best_results['Recall'] = best_results['Recall'].round(2)
best_results['F1 Score'] = best_results['F1 Score'].round(2)
best_results['ROC AUC'] = best_results['ROC AUC'].round(2)
best_results['Jaccard Score'] = best_results['Jaccard Score'].round(2)
all_results['Precision'] = all_results['Precision'].round(2)
all_results['Recall'] = all_results['Recall'].round(2)
all_results['F1 Score'] = all_results['F1 Score'].round(2)
all_results['ROC AUC'] = all_results['ROC AUC'].round(2)
all_results['Jaccard Score'] = all_results['Jaccard Score'].round(2)


# Create a list of available metrics
metrics = ['Precision', 'Recall', 'F1 Score', 'ROC AUC', 'Jaccard Score', 'Net Savings ($)']

# Create a list of unique model names for selection
model_names = best_results['Model Name'].unique()

# Initialize the Dash app
app = dash.Dash(__name__)

# Define the layout
app.layout = html.Div([
    html.H1("Model Evaluation Dashboard on Unseen Test Data"),
    
    # Dropdown for selecting a metric to display
    dcc.Dropdown(
        id='metric-selector',
        options=[{'label': metric, 'value': metric} for metric in metrics],
        value='Precision',  # Default value
        clearable=False
    ),
    
    # Bar chart to show the selected metric vs. Threshold
    dcc.Graph(id='metric-graph'),
    
    # Div to hold the two pie charts side by side
    html.Div([
        dcc.Graph(id='confusion-matrix-pie', style={'display': 'inline-block', 'width': '49%'}),
        dcc.Graph(id='cost-pie-chart', style={'display': 'inline-block', 'width': '49%'})
    ]),
    
    # Table to display the detailed metrics for the top models
    dash_table.DataTable(
        id='model-details-table',
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'center',
            'minWidth': '100px',
            'width': '100px',
            'maxWidth': '100px',
            'whiteSpace': 'normal'
        }
    ),
    
    # Dropdown for selecting a model to display its metrics
    html.H2("Select a Model to View Detailed Metrics"),
    dcc.Dropdown(
        id='model-selector',
        options=[{'label': name, 'value': name} for name in model_names],
        value=model_names[0],  # Default value is the first model
        clearable=False
    ),
    
    # Table to display the selected model's metrics
    dash_table.DataTable(
        id='selected-model-details-table',
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'center',
            'minWidth': '100px',
            'width': '100px',
            'maxWidth': '100px',
            'whiteSpace': 'normal'
        },
        page_size=10  # Allows pagination
    )
])

# Callback to update the graph and table based on selected metric
@app.callback(
    Output('metric-graph', 'figure'),
    Output('confusion-matrix-pie', 'figure'),
    Output('cost-pie-chart', 'figure'),
    Output('model-details-table', 'data'),
    Output('model-details-table', 'columns'),
    Output('selected-model-details-table', 'data'),
    Output('selected-model-details-table', 'columns'),
    Input('metric-selector', 'value'),
    Input('model-selector', 'value')
)
def update_graph_and_table(selected_metric, selected_model):
    if not selected_metric or not selected_model:
        raise dash.exceptions.PreventUpdate

    # Filter the DataFrame to get the top 3 models based on the selected metric
    top_models = best_results.nlargest(3, selected_metric)
    
    # Create the bar plot for the selected metric
    fig_metric = px.bar(top_models, x='Threshold', y=selected_metric, color='Model Name', 
                        title=f"Top 3 Models by {selected_metric}", barmode='group')
    
    # Select the best model to display confusion matrix and cost pie
    best_model = top_models.iloc[0]
    
    # Create a pie chart for the confusion matrix without True Negatives
    labels = ['True Positives (TP)', 'False Positives (FP)', 'False Negatives (FN)']
    values = [best_model['True Positives (TP)'], best_model['False Positives (FP)'], best_model['False Negatives (FN)']]
    
    fig_pie = go.Figure(data=[go.Pie(labels=labels, values=values, hole=.3)])
    fig_pie.update_layout(
        title=f"Confusion Matrix for {best_model['Model Name']} at Threshold {best_model['Threshold']:.2f}",
        title_font_size=12
    )
    
    # Create a pie chart for cost of operation and net savings
    cost_labels = ['Cost of Operation ($)', 'Net Savings ($)']
    cost_values = [best_model['Cost of Operation ($)'], best_model['Net Savings ($)']]
    
    fig_cost_pie = go.Figure(data=[go.Pie(labels=cost_labels, values=cost_values, hole=.3)])
    fig_cost_pie.update_layout(
        title=f"Total Cost for {best_model['Model Name']} at Threshold {best_model['Threshold']:.2f}",
        title_font_size=12
    )
    
    # Prepare data and columns for the detailed metrics table
    model_details_data = top_models.to_dict('records')
    model_details_columns = [{"name": i, "id": i} for i in top_models.columns]
    
    # Filter the DataFrame to get the metrics for all records of the selected model
    selected_model_metrics = all_results[all_results['Model Name'] == selected_model]
    selected_model_data = selected_model_metrics.to_dict('records')
    selected_model_columns = [{"name": i, "id": i} for i in selected_model_metrics.columns]
    
    return (fig_metric, fig_pie, fig_cost_pie, 
            model_details_data, model_details_columns, 
            selected_model_data, selected_model_columns)

# Run the app
app.run_server(mode='inline')

# Run the app in your local environment
To run your dashboard as a well dashboard in html format: of course it will only work if you have the app or not, then you need to create the app.
1. Open anaconda prompt
2. nagivate to your directory that has the app.py: cd \Users\Black\DS\TIP\fraud-detection\notebooks
3. run the app: python model_dashboard.py
4. open your browser and paste this: http://127.0.0.1:8060/   or  just click this link to run from the notebook directly http://127.0.0.1:8050/ - change the ports depending on the port in the app or change to avoid conflict with the local port